#### **Data Dictionary**

| Column Name                   | Data Type                    | Description                                                                                                                         |
| ----------------------------- | ---------------------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| **transaction_id**            | String (Object)              | Unique identifier assigned to each transaction. Used to track and reference individual payment records.                             |
| **customer_id**               | String (Object)              | Unique identifier for the customer initiating the transaction. Enables analysis of customer behavior across multiple transactions.  |
| **timestamp**                 | Date-Time (stored as String) | Date and time when the transaction was initiated or processed. Useful for analyzing transaction timing and behavioral patterns.     |
| **home_country**              | Categorical (String)         | Customer's registered country of residence or account registration country.                                                         |
| **source_currency**           | Categorical (String)         | Currency from which funds are being sent.                                                                                           |
| **dest_currency**             | Categorical (String)         | Currency that the recipient receives after conversion.                                                                              |
| **channel**                   | Categorical (String)         | Platform or method used to initiate the transaction, such as mobile application, web portal, or other payment channels.             |
| **amount_src**                | String (Object)              | Original transaction amount in the source currency before conversion. May require conversion to numeric format for analysis.        |
| **amount_usd**                | Float                        | Transaction amount standardized to U.S. dollars for consistent analysis across different currencies.                                |
| **fee**                       | Float                        | Transaction processing fee charged by NovaPay.                                                                                      |
| **exchange_rate_src_to_dest** | Float                        | Exchange rate applied to convert the source currency into the destination currency.                                                 |
| **device_id**                 | String (Object)              | Unique identifier of the device used to perform the transaction.                                                                    |
| **new_device**                | Boolean                      | Indicates whether the transaction originated from a device not previously associated with the customer account.                     |
| **ip_address**                | String (Object)              | IP address from which the transaction was initiated.                                                                                |
| **ip_country**                | Categorical (String)         | Country associated with the originating IP address.                                                                                 |
| **location_mismatch**         | Boolean                      | Indicates whether the customer's registered country differs from the country identified through the IP address.                     |
| **ip_risk_score**             | Float                        | Risk score assigned to the originating IP address based on predefined risk indicators. Higher values represent greater risk.        |
| **kyc_tier**                  | Categorical (String)         | Customer verification level under Know Your Customer (KYC) requirements.                                                            |
| **account_age_days**          | Integer                      | Number of days since the customer account was created.                                                                              |
| **device_trust_score**        | Float                        | Trust score assigned to the device based on historical activity and usage patterns. Higher values indicate greater trustworthiness. |
| **chargeback_history_count**  | Integer                      | Number of historical chargebacks or disputed transactions associated with the customer account.                                     |
| **risk_score_internal**       | Float                        | Internal risk score generated by NovaPay's existing fraud monitoring controls.                                                      |
| **txn_velocity_1h**           | Integer                      | Number of transactions performed by the customer within the previous one-hour period.                                               |
| **txn_velocity_24h**          | Integer                      | Number of transactions performed by the customer within the previous twenty-four-hour period.                                       |
| **corridor_risk**             | Float                        | Risk score associated with the transaction corridor, typically based on source and destination country combinations.                |
| **is_fraud**                  | Integer (Binary: 0/1)        | Target variable indicating whether a transaction was confirmed as fraudulent (1) or legitimate (0).                                 |


#### **Import libraries**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### **Data quality check**

In [3]:
df = pd.read_csv("nova_pay_combined.csv")

In [13]:
df.shape

(11400, 26)

In [5]:
df.head(3)

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11400 entries, 0 to 11399
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             11400 non-null  object 
 1   customer_id                11400 non-null  object 
 2   timestamp                  11371 non-null  object 
 3   home_country               11400 non-null  object 
 4   source_currency            11400 non-null  object 
 5   dest_currency              11400 non-null  object 
 6   channel                    11400 non-null  object 
 7   amount_src                 11400 non-null  object 
 8   amount_usd                 11095 non-null  float64
 9   fee                        11105 non-null  float64
 10  exchange_rate_src_to_dest  11400 non-null  float64
 11  device_id                  11400 non-null  object 
 12  new_device                 11400 non-null  bool   
 13  ip_address                 11095 non-null  obj

##### Summary
- Timestamp is currently an object so need to be converted to a **datetime**
- amount_src is currently an object so need to be converted to a **float**

In [8]:
df.describe()

,amount_usd,fee,exchange_rate_src_to_dest,ip_risk_score,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
count,11095.000000,11105.000000,11400.000000,11400.000000,11400.000000,11105.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000,11400.000000
mean,452.022083,100.309441,167.540397,0.396726,393.793158,0.653681,0.048509,0.267134,0.458333,0.723509,0.045501,0.087456
std,1403.973062,958.128504,382.023827,0.270507,342.348393,0.273012,0.256194,0.142983,1.524494,1.958390,0.084942,0.282515
min,7.230000,-1.000000,0.592000,0.004000,1.000000,-0.100000,0.000000,0.000000,-1.000000,0.000000,0.000000,0.000000
25%,92.465000,2.380000,1.000000,0.209000,147.000000,0.515000,0.000000,0.169000,0.000000,0.000000,0.000000,0.000000
50%,163.480000,3.500000,7.142857,0.325000,298.000000,0.658000,0.000000,0.223000,0.000000,0.000000,0.000000,0.000000
75%,302.190000,5.550000,73.529412,0.487000,661.000000,0.894000,0.000000,0.391000,0.000000,0.000000,0.050000,0.000000
max,12498.570000,9999.990000,1388.888889,1.200000,1095.000000,0.999000,2.000000,0.900000,8.000000,11.000000,0.250000,1.000000


In [17]:
df.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [18]:
df.duplicated().sum()

np.int64(200)